# Decision Tree Development Notebook (v1) 

In [1]:
import numpy as np
import pandas as pd
from collections import Counter
from functools import reduce
from typing import Literal
from datetime import datetime, timedelta
import time

In [2]:
# UTILS

def prod(arr: list) -> float:
    """Computes the product of all elevents of a given vector"""
    return reduce(lambda x, y: x * y, arr, 1)

def gini(arr: list) -> float:
    """Computes Gini Impurity for a given label vector"""
    n = len(arr)
    value_counts = dict(Counter(arr))
    unique_categories = list(value_counts.keys())
    prob_list = {cat: (value_counts[cat] / n) for cat in unique_categories}
    return 1 - sum(p**2 for p in prob_list.values())

def get_cutoff(feature, y, base_gini, return_gain: bool = False):
    """Searches a good cutoff in a feature vector in order to maximize node purity gain"""
    feature_nrows_unique = len(set(feature))
    # cuts = int(np.floor(np.min([feature_nrows_unique, 10])))
    CUTS = 10
    if feature_nrows_unique <= CUTS:
        candidates = np.unique(feature)
    else:
        candidates = np.unique([np.quantile(feature, q=q).item() for q in np.linspace(0, 1, CUTS)])
        
    curr_winner, curr_winner_gain = None, 0
    for candidate in candidates:
        left_gini, right_gini = gini(y[feature < candidate]), gini(y[feature >= candidate])
        gain = (base_gini - left_gini) + (base_gini - right_gini)  # TODO: Generalize
        if gain > curr_winner_gain:
            curr_winner_gain, curr_winner = gain, candidate
    return curr_winner if return_gain is False else (curr_winner, curr_winner_gain)

def rates_by_category(y, cats: list[int]):
    """Retrieves the percentage of data that corresponds to each class/category"""
    return [(y == cat).mean().item() for cat in cats]


In [3]:
class BinaryTree:
    """Manual implementation of a BinaryTree"""
    def __init__(
        self,
        gini: float = None,
        proba: list[float] = None,
        chained_conditions: list[tuple[int, str, float]] = None,
        depth: int = 1
    ):
        # Data structure info
        self.left = None  # <Less than> child
        self.right = None  # <Greater than or equal to> child
        self.parent = None  # To comeback
        self.depth: int = depth
        
        # Training state
        self.feature_index: int = None  # Index of the feature used to split current node
        self.cutoff_value: float = None  # Boundary
        
        self.gini: float = gini  # Impurity
        self.proba: list[float] = proba  # List of probabilities (not calibrated yet)
        self.chained_conditions: list[tuple[int, str, float]] = chained_conditions  # Necessary conditions to arrive to the node
        if chained_conditions is None:
            self.chained_conditions = []

        self.branch_status: Literal["OK"] = None

    def _set_parent(self, parent) -> None:
        self.parent = parent
        if hasattr(self.parent, "chained_conditions"):
            if self.parent.chained_conditions is not None:
                self.chained_conditions = self.parent.chained_conditions + self.chained_conditions
        if hasattr(self.parent, "depth"):
            self.depth = self.parent.depth + 1

    def _set_child(self, child, direction: Literal["left", "right"]) -> None:
        assert direction in ["left", "right"], "<direction> value must be either left or right"
        if direction == "left":
            self.left = child
        else:
            self.right = child
        child._set_parent(self)

    def _set_children(self, children: list) -> None:
        assert len(children) == 2, f"Number of children must be 2"
        child_left, child_right = children[0], children[1]
        self._set_child(child_left, "left")
        self._set_child(child_right, "right")

    def _set_branch_status(self, branch_status: Literal["OK"]) -> None:
        self.branch_status = branch_status

    def _is_leaf(self):
        return (self.left is None) and (self.right is None)

    def __repr__(self):
        str1 = "<BinaryTree object>"
        str2 = f"Left child = {type(self.left)}"
        str3 = f"Right child = {type(self.right)}"
        str4 = f"Feature index (on train data) = {self.feature_index}"
        str5 = f"Cutoff value in feature vector = {self.cutoff_value}"
        str6 = f"Impurity = {self.gini}"
        str7 = f"Mean label = {self.proba}"
        str8 = f"Parent node = {type(self.parent)}"
        str9 = f"Node conditions = {self.chained_conditions}"
        str10 = f"<Branch status: {self.branch_status}>"
        final_str = "\n - ".join([str1, str2, str3, str4, str5, str6, str7, str8, str9, str10])
        return final_str

    @staticmethod
    def filter_data(data, queries: list[tuple[int, str, float]], reference_data):
        X = reference_data  # Function-scope data
        if len(queries) > 0:
            query_str = " & ".join([f"(X[:, {t[0]}] {t[1]} {t[2]})" for t in queries])
            str_to_eval = f"data[{query_str}]"
            return eval(str_to_eval)
        return data

In [4]:
class DecisionTreeClassifier:
    """Manual implementation of DecisionTreeClassifier"""
    def __init__(self):
        self.tree = BinaryTree()
        self.trained: bool = False
        self.max_depth: int = None
        self.train_time_sec: int = None
        self.train_time: str = None


    def train(self, X, y) -> None:
        """
        Executes the training process, note that this is an inplace logic
        Reminder: stop criteria is gini_of_node = 0
        """
        if self.trained is True:
            raise ValueError("Tree is already trained")

        # If Pandas...
        if hasattr(X, "values"):
            X = X.values
        if hasattr(y, "values"):
            y = y.values

        n_features = X.shape[1]
        cats = sorted(list(set(y)))
        max_depth = 0

        t1 = time.time()
        subtree = self.tree  # Start from root
        while self.tree.branch_status != "OK":

            if hasattr(subtree, "depth"):
                if subtree.depth > max_depth:
                    max_depth = subtree.depth

            if (subtree.left is not None) and (subtree.right is not None):
                if (subtree.left.branch_status == "OK") and (subtree.right.branch_status == "OK"):
                    subtree._set_branch_status("OK")
                    subtree = subtree.parent
                    continue

            if subtree.feature_index is None:  # Untrained node proxy flag
                X_subtree = self.tree.filter_data(X, queries=subtree.chained_conditions, reference_data=X)
                y_subtree = self.tree.filter_data(y, queries=subtree.chained_conditions, reference_data=X)
                subtree_impurity = gini(y_subtree)
                
                purity_gains = []
                for feature_index in range(n_features):
                    cutoff_value, purity_gain = get_cutoff(X_subtree[:, feature_index], y_subtree, base_gini=subtree_impurity, return_gain=True)
                    purity_gains.append((feature_index, cutoff_value, purity_gain))
                best_feature_index, best_cutoff_value, _ = max(purity_gains, key=lambda t: t[2])

                # Set split info
                subtree.feature_index = best_feature_index
                subtree.cutoff_value = best_cutoff_value.item()
                subtree.gini = subtree_impurity
                subtree.proba = rates_by_category(y_subtree, cats)

                # Feature space subset
                y_left = y_subtree[X_subtree[:, best_feature_index] < best_cutoff_value]
                y_right = y_subtree[X_subtree[:, best_feature_index] >= best_cutoff_value]

                child_left = BinaryTree(
                    gini=gini(y_left),
                    proba=rates_by_category(y_left, cats),
                    chained_conditions=[(best_feature_index, "<", best_cutoff_value.item())],
                )
                child_right = BinaryTree(
                    gini=gini(y_right),
                    proba=rates_by_category(y_right, cats),
                    chained_conditions=[(best_feature_index, ">=", best_cutoff_value.item())]
                )
                subtree._set_children([child_left, child_right])

            if child_left.gini == 0:
                child_left._set_branch_status("OK")
            if child_right.gini == 0:
                child_right._set_branch_status("OK")
            
            if (subtree.left.branch_status == "OK") and (subtree.right.branch_status == "OK"):
                subtree._set_branch_status("OK")
                subtree = subtree.parent
            elif subtree.left.branch_status != "OK":
                subtree = subtree.left
            elif subtree.right.branch_status != "OK":
                subtree = subtree.right

        self.trained = True
        if self.max_depth is None:
            self.max_depth = max_depth

        t2 = time.time()
        self.train_time_sec = t2 - t1
        self.train_time = str(timedelta(seconds=self.train_time_sec))


    def _single_inference(self, x) -> list[float]:
        subtree = self.tree
        while True:
            feature_index = subtree.feature_index
            cutoff_value = subtree.cutoff_value
            boolean = (x[feature_index] >= cutoff_value)
            if boolean:
                subtree = subtree.right
            else:
                subtree = subtree.left
            if subtree._is_leaf():
                proba = subtree.proba
                break
        return proba


    def inference(self, X) -> list[list[float]]:
        if not hasattr(X, "shape"):
            X = np.array(X)
        if len(X.shape) >= 2:
            return np.array([self._single_inference(x) for x in X])
        else:
            return np.array(self._single_inference(X))
    

In [5]:
dtc = DecisionTreeClassifier()
data = pd.read_parquet("../example_data/iris_dataset.parquet")
data

,sepal_length_cm,sepal_width_cm,petal_length_cm,petal_width_cm,target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


In [6]:
data = data.sample(frac=1).reset_index(drop=True)

In [7]:
train_data = data.iloc[:130, :]
test_data = data.iloc[130:, :]

In [8]:
X_train, y_train = train_data.drop(columns=["target"]).values, train_data["target"].values
X_test, y_test = test_data.drop(columns=["target"]).values, test_data["target"].values

In [9]:
model = DecisionTreeClassifier()

In [10]:
model.train(X_train, y_train)

In [11]:
inference = model.inference(X_test)

In [12]:
y_pred = np.array([list(vec).index(1.) for vec in inference])
y_pred

array([2, 2, 2, 2, 1, 0, 2, 1, 1, 0, 1, 1, 1, 0, 2, 2, 2, 2, 1, 1])

In [13]:
y_test

array([2, 2, 2, 2, 1, 0, 2, 1, 1, 0, 1, 1, 1, 0, 2, 2, 2, 2, 1, 2])

In [14]:
results = pd.DataFrame({"y_test": y_test, "y_pred": y_pred})

In [15]:
z = pd.crosstab(results["y_test"], results["y_pred"])

In [16]:
z

y_pred,0,1,2
y_test,,,
0,3,0,0
1,0,7,0
2,0,1,9
